In [1]:
!pip install sentence-transformers pandas tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from tqdm import tqdm

In [5]:
ROOT_DIR = "/content/drive/MyDrive/Resilio/outputs"
SAVE_DIR = "/content/drive/MyDrive/Resilio/outputs/response_cossim_results"

In [6]:
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
def cos_sim(a, b):
    # Vectorize and calculate cosine similarity percentage
    emb = model.encode([a, b], normalize_embeddings=True, show_progress_bar=False)
    return float(cosine_similarity([emb[0]], [emb[1]])[0][0]) * 100

In [8]:
def process_all_directories():
    # Gather all responses.json paths
    json_files = []
    for root, dirs, files in os.walk(ROOT_DIR):
        if "responses.json" in files:
            # We want to exclude the final SAVE_DIR if it exists
            if SAVE_DIR not in root:
                json_files.append(os.path.join(root, "responses.json"))

    if not json_files:
        print("No responses.json files found in the specified directory structure.")
        return

    print(f"Found {len(json_files)} datasets to process.")

    for file_path in json_files:
        # Extract metadata from path: ./provider/model_quant/category/responses.json
        parts = file_path.split(os.sep)
        # Assuming path is like: ROOT/provider/model_quant/category/responses.json
        # Adjust indices if your root depth varies
        category = parts[-2]
        model_quant = parts[-3]
        provider = parts[-4]

        print(f"\nProcessing: {provider} | {model_quant} | {category}")

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 1. Organize: responses[base_id][drift_level][variant]
        responses_map = defaultdict(lambda: defaultdict(dict))
        for item in data:
            base = item["base_id"]
            drift = item["drift_level"]
            variant = item["variant"]
            resp = item.get("response", "")
            responses_map[base][drift][variant] = resp

        # 2. Get sorted base IDs (qna_1 to qna_10, etc.)
        bases = sorted(responses_map.keys(), key=lambda x: int(x.split("_")[1]))

        # 3. Calculate similarities for each drift level (1-5)
        rows = []
        for drift in tqdm(range(1, 6), desc=f" Calculating {category}"):
            row = {"Drift_Level": drift}

            for base in bases:
                # Baseline is always Drift 0, Variant 0
                ideal_text = responses_map[base][0].get(0, "")

                if not ideal_text:
                    continue

                for v in [1, 2, 3]:
                    drift_text = responses_map[base][drift].get(v, "")

                    if not drift_text or not ideal_text:
                        sim = np.nan
                    else:
                        sim = cos_sim(ideal_text, drift_text)

                    row[f"{base}_V{v}"] = round(sim, 2) if not np.isnan(sim) else None
            rows.append(row)

        # 4. Save results
        # Path: SAVE_DIR/provider/model_quant/category_cossim.csv
        out_folder = os.path.join(SAVE_DIR, provider, model_quant)
        os.makedirs(out_folder, exist_ok=True)

        df = pd.DataFrame(rows)
        out_name = f"{category}_response_cossim.csv"
        df.to_csv(os.path.join(out_folder, out_name), index=False)

    print(f"\nAll processing complete. Results saved in '{SAVE_DIR}' folder.")

In [9]:
process_all_directories()

Found 36 datasets to process.

Processing: mistralai | Mistral-7B-Instruct-v0.2_4bit | reasoning


 Calculating reasoning: 100%|██████████| 5/5 [00:16<00:00,  3.34s/it]



Processing: mistralai | Mistral-7B-Instruct-v0.2_4bit | classification


 Calculating classification: 100%|██████████| 5/5 [00:08<00:00,  1.78s/it]



Processing: mistralai | Mistral-7B-Instruct-v0.2_4bit | logical


...


Processing: microsoft | Phi-3-mini-4k-instruct_fp16 | reasoning


 Calculating reasoning: 100%|██████████| 5/5 [00:16<00:00,  3.31s/it]



Processing: microsoft | Phi-3-mini-4k-instruct_fp16 | qna


 Calculating qna: 100%|██████████| 5/5 [00:16<00:00,  3.35s/it]


All processing complete. Results saved in '/content/drive/MyDrive/Resilio/outputs/response_cossim_results' folder.
